In this notebook we extract plain text from original PDF-format reports, with specific focus on extracting the "Government Policies and Programs" sections, before saving them to plain-text (TXT) files. The workflow is designed as follows:
1. Retain only body of original reports (from title to reference), excluding the cover (1-2 pages) and attached tables.
    - **Approach**: start from the page containing "The Mineral Industry of" or "The Mineral Industries of" (case-sensitive) and end before the first co-appearance of "table" and ": production of mine" (or ": estimated production of mine") (case-insensitive).
2. Analyze key properties of each block in the file.
    - **Approach**: PyMuPDF can decompose PDF pages into "blocks" of contents with formats (through `page.get_text("dict")` for detailed layouts and `page.get_text("blocks")` for plain text). We observed that **main text in USGS reports are in font size 10**, distinct from that of title, footnote/reference and footers (the latter two in font size 8 or 9). Thus we developed a module to identify which categor(ies) of contents the block belong to (`header/footer/footnote/reference/main-body/section-title/section-subtitle`) by calculating the (1) (rounded) dominant font size, (2) dominant font style and (3) in-page position of each block. The module is encapsulated in `block_format()` function in `/modules/extract_text/BlockFont.py`, referring to which for detailed criteria (and corresponding underlying assumptions).
    - **Assumptions on block classification**: (1) `is_mainbody`: Dominant font size == 10; (2) `is_header`: Dominant font size > 10; (3) `is_footer`: Dominant font size < 10 while locates at the bottom of the page ($y$-coordinate of upper-left corner > 700); (4) `is_footnote`: Start with a superscript (span) whose font size < 7 and dominant font size < 10; (5) `is_reference`: Dominant font size < 10 while not belongs to footer or footnote; (6) `is_sectitle`: Dominant font size == 10 while dominant font style contains "Bold" BUT NOT "Ital" (*thus a subset of main-body*); (7) `is_secsubtitle`: Dominant font size == 10 while dominant font style contains BOTH "Bold" AND "Ital" (*thus also a subset of main-body*).
3. Retain and sort main-body and reference blocks by column (left to right in two-column layout) and identify the sectional affiliation of each block.
    - **Preliminary Outline [undergone huge amendments]**: (1) Mark the retained blocks on each page $\Rightarrow$ (2) Calculate the horizontal ($x$) coordinate for central axis of each page, and read from top to bottom first the left column then the right column $\Rightarrow$ (3) Examining the dominant typeface of each block and mark those with bold font (title for each section) $\Rightarrow$ (4) Write by sequence blocks into a same string (KEY ISSUE: dealing with paragraphs spanning two pages), and split the string by identified section titles.
    - **Approach [final version]**: (1) Invoke `block_format()` function from `BlockFont.py` to extract first-line indentation, in-page position and identify the categor(ies) which blocks belong to. (2) Remove blocks as headers (`is_header == False`), footers (`is_footer == False`) or footnotes (`is_footnote == False`); and retain only blocks as main-body (`is_mainbody == True`) or references (`is_reference == True`). (3) Calculate the $x$-coordinate of **central axis** on each page (to seperate left and right columns of contents), which threads through the midpoint of the left boundary of the right column of blocks and right boundary of the left column. (3) After **creating a dataframe storing all properties (metadata) of the blocks**, group the blocks of main-bodies and references by column on each page (based on whether the upper-left corner of the block locates left or right to the central axis), and sort blocks in each column by top-down by the $y$-coordinates of their upper-left corner. (4) Match sorted metadata of blocks on each page with plain text in each block, and concatenate sorted rows of data from different pages to obtain a cleansed full list of main-body and reference blocks in the PDF file. (5) Treat rows where `is_sectitle == True` in the metadata dataframe as cutpoints, based on which group blocks by section, and assign a new property `section_name` to all blocks in each group with the value of `contents` in the first row of the group (typically the block of section-title, except for the introductory section in some PDF files where section-title is missing).
4. Retain blocks affiliated to "Government Policies and Programs" section and save extracted plain text (prettified and joined) into TXT file.
    - **Approach**: Query for rows where `section_name` contains "Government Policies". Then remove the section-title block (`is_sectitle == True`) and join main-body texts by paragraph. Typically a block identified by PyMuPDF corresponds to a paragraph (except for special layouts such as in reference section), but it could be a little tricky for paragraphs spanning across columns (whatever across the left and right columns on a same page or across pages). But truncated paragraphs do have certain features, mainly the first-line indentation of the block. If a non-section-title block is not first-line-indented, then it should be either part of a truncated paragraph or a substitle of the section. We utilize this observed feature to join text in blocks spanning across columns with a blank while in other cases join text in blocks with a line break. Note before joining two slices of texts, all line breaks within text from an individual block are removed to prettify the output layout.

#### Sec. 1: Pick a PDF and retain remove covers and appendices (attached tables) (provisional iteration indicator $i$ introduced)

In [ ]:
import pymupdf
import os
import numpy as np
import pandas as pd
import sys
# Add /modules/extract_text directories to where packages could be imported
sys.path.append("../../modules/extract_text")
import BlockFont as bf

# Import PDF
pdf_dir = "../../data/original_report/usgs_africa/"
pdf_names = os.listdir(pdf_dir)
pdf_paths = [os.path.join(pdf_dir, pdf_name) for pdf_name in pdf_names]
# Select a PDF file for experiment (REPLACE WITH FOR-LOOP LATER)
# ------------Provisional Iteration Indicator------------
i = 579 # Select the file in the directory
# -------------------------------------------------------
pdf_path = pdf_paths[i]
pdf = pymupdf.open(pdf_path)

# Retain only pages in the main body of the file (remove covers and appendices)
start_page_num = 0
end_page_num = len(pdf) # IMPORTANT: Variable 'end_page_num' records the page number after the last page. But since range() function exactly excludes the redundant page when generating a sequence, this index well fits our end.
for page in pdf:
    page_text = page.get_text('text')
    if page_text.count("The Mineral Industry of") != 0 or page_text.count("The Mineral Industries of"): # Case-sensitive, where the main body starts
        start_page_num = page.number
    # CRITICAL DEBUG: (1) Previous discriminant "page_text.count("TABLE") != 0" could fail as in some reports appendices turn out start with "tABLe" or other bizarre composition of cases. But directly search for case-insensitive 'table' could mistake reference to appendices in the main text, so we try to add another condition, detecting the joint occurrence of two phrases.
    # (2) Australia 2007 Report misspelled "mineral" on the first page of the appendices as "mineal"
    if len(page.search_for('table')) !=0 and (len(page.search_for(f': production of mine')) or len(page.search_for(f': estimated production of mine'))) !=0: # Case-insensitive, the page after where the main body ends
        end_page_num = page.number
        break
if start_page_num == end_page_num:
    print("Bizarre format! Annexed table is embedded in the 1st page of the main body.") 
    # IMPORTANT: In for-loop we skip the current round here with 'continue' command if such issue is detected
body_range = list(range(start_page_num, end_page_num))
pdf.select(body_range) # IMPORTANT: Different from ordinary python commands which make a copy of the original object, this command directly amends the object "pdf". So DO NOT write "pdf = pdf.select(...)".
# Note: Up till now, all page indexes matches the page numbers in the original PDF file.

In [3]:
# FOR DEBUGGING
body_range

[1, 2]

#### Sec. 2: Iterate through pages in the PDF to create a property matrix of all main-text and reference blocks in the PDF file, sorted by human-reading sequence (loop indicator $j$ introduced)

In [4]:
# Iterate through pages in the PDF
is_firstpage = True # Indicator for whether it's first page, correspondingly whether to create or concatenate with the aggregate dataframe
allpages_mainbody_wizref_properties_df = pd.DataFrame()
for j in range(len(pdf)):
    # Extract all blocks on the page
    blocks_dict_list = pdf[j].get_text("dict")['blocks'] # Extract plain text in blocks on the page
    blocks_list = pdf[j].get_text("blocks") # Extract detailed format info of blocks on the page
    # ------------Provisonal Skip-------------
    if len(blocks_list) == 0: # IMPORTANT: Skip the current page if it turns out to be blank.
        continue
    # ----------------------------------------

    # Retrieve properties of all blocks
    all_blocks_properties_df = pd.DataFrame([bf.block_format(block_dict) for block_dict in blocks_dict_list])
    all_blocks_properties_df['page_in_body'] = j
    # Only retain blocks belongs to main-text and reference categories
    mainbody_wizref_properties_df = all_blocks_properties_df.query("not (is_header or is_footer or is_footnote)")

    ## Figure out the x-coordinate for central axis, based on which the page is split into two columns
    # Calculate the x-coordinate for left boundary of the right column and vice versa. 
    rightcolumn_start_x_coordinate = min([x_coord for x_coord in mainbody_wizref_properties_df['upperleft_x_coordinate'] if x_coord > 300])
    leftcolumn_end_x_coordinate = max([x_coord for x_coord in mainbody_wizref_properties_df['lowerright_x_coordinate'] if x_coord < rightcolumn_start_x_coordinate])
    # Central axis is set as threading through the midpoint of these two coordinates. 
    central_axis_x_coordinate = (leftcolumn_end_x_coordinate + rightcolumn_start_x_coordinate)/2

    ## Group main-text and reference blocks by column, and sort blocks in each group by y-coordinate, before rejoining them in the dataframe by row
    # Left-column blocks sorting and indexing
    leftcolumn_blocks_properties_df = mainbody_wizref_properties_df.query("upperleft_x_coordinate < @central_axis_x_coordinate") # IMPORTANT: The selected coordinate should be the same for two discriminants to avoid rare occassions where a long banner horizonally spans across the central axis.
    leftcolumn_blocks_properties_df['is_right_column'] = 0 # Whether current column is the right column on the page
    leftcolumn_blocks_properties_df['index_in_column'] = list(range(len(leftcolumn_blocks_properties_df))) # Index the blocks within the column
    # Right-column blocks sorting and indexing
    rightcolumn_blocks_properties_df = mainbody_wizref_properties_df.query("upperleft_x_coordinate > @central_axis_x_coordinate")
    rightcolumn_blocks_properties_df['is_right_column'] = 1 # Whether current column is the right column on the page
    rightcolumn_blocks_properties_df['index_in_column'] = list(range(len(rightcolumn_blocks_properties_df))) # Index the blocks within the column
    # Reconcatenate the two sorted property matrices by row, and save back to variable 'mainbody_wizref_properties_df'.
    mainbody_wizref_properties_df = pd.concat([
        leftcolumn_blocks_properties_df.sort_values(by="upperleft_y_coordinate"),
        rightcolumn_blocks_properties_df.sort_values(by="upperleft_y_coordinate")
    ], axis=0)

    ## Match block property matrix with block plain text table by column "index_in_page", and save to variable "mainbody_wizref_fulltext_df"
    blocks_list_df = pd.DataFrame({
        'index_in_page': [block[5] for block in blocks_list],
        'contents': [block[4] for block in blocks_list]
    })
    mainbody_wizref_fulltext_df = pd.merge(mainbody_wizref_properties_df, blocks_list_df, on='index_in_page', how="left")

    ## Create or concatenate to variable 'allpages_mainbody_wizref_properties_df'
    if is_firstpage:
        allpages_mainbody_wizref_properties_df = mainbody_wizref_fulltext_df.copy()
        is_firstpage = False # All the pages afterwards would not be the first page
    else:
        allpages_mainbody_wizref_properties_df = pd.concat([allpages_mainbody_wizref_properties_df, mainbody_wizref_fulltext_df], axis=0)

## Determine the sectional affiliation of each block in the PDF file
# .cumsum() function allows us to assign a unique index for blocks within each section, as it creates a list where each item is the sum of all previous elements (and boolean values are mapped to 0-1 dichotomy)
allpages_mainbody_wizref_properties_df['section_num'] = allpages_mainbody_wizref_properties_df['is_sec_title'].cumsum()
# We mask the section_num == 0 segment to exclude introduction section (in most occassions without a title). Then we group the rows by the section they belong to and assign the value of "contents" in the first row (i.e., section title) to the section_name property of all blocks in the same segment (using .group_by("section_num").transform("first") function).
allpages_mainbody_wizref_properties_df['section_name'] = allpages_mainbody_wizref_properties_df.mask(allpages_mainbody_wizref_properties_df['section_num']==0).groupby('section_num')['contents'].transform('first') 

In [5]:
# FOR DEBUGGING: # FOR DEBUGGING: Property matrix of all blocks (regardless of category) ON current page (where the loop stops)
all_blocks_properties_df

,index_in_page,is_firstline_indented,dominant_font_size,dominant_font_style,is_header,is_footer,is_footnote,is_reference,is_mainbody,is_sec_title,is_sec_subtitle,upperleft_x_coordinate,upperleft_y_coordinate,lowerright_x_coordinate,lowerright_y_coordinate,page_in_body
0,0,False,8,Times-Roman,False,True,False,False,False,False,False,36.000000,748.071960,573.999512,757.000000,0
1,1,False,17,TimesNewRomanPS-BoldMT-S,True,False,False,False,False,False,False,131.077499,27.629906,480.920654,59.621906,0
2,2,False,15,TimesNewRomanPSMT,True,False,False,False,False,False,False,243.425507,65.148911,368.585541,84.858910,0
3,3,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,100.163582,286.790039,149.303589,0
4,4,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,148.163574,293.740051,305.303619,0
5,5,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,False,36.000000,315.973602,83.600006,329.303619,0
6,6,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,340.163605,291.780029,401.303619,0
7,7,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,False,36.000000,411.973602,180.250000,425.303619,0
8,8,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,436.163605,293.350037,497.303619,0
9,9,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,False,316.799988,99.812386,400.949951,113.142387,0


In [6]:
# FOR DEBUGGING: Property matrix of only mainbody and reference blocks ON current page (where the loop stops)
mainbody_wizref_properties_df

,index_in_page,is_firstline_indented,dominant_font_size,dominant_font_style,is_header,is_footer,is_footnote,is_reference,is_mainbody,is_sec_title,is_sec_subtitle,upperleft_x_coordinate,upperleft_y_coordinate,lowerright_x_coordinate,lowerright_y_coordinate,page_in_body,is_right_column,index_in_column
3,3,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,100.163582,286.790039,149.303589,0,0,0
4,4,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,148.163574,293.740051,305.303619,0,0,1
5,5,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,False,36.000000,315.973602,83.600006,329.303619,0,0,2
6,6,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,340.163605,291.780029,401.303619,0,0,3
7,7,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,False,36.000000,411.973602,180.250000,425.303619,0,0,4
8,8,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,36.000000,436.163605,293.350037,497.303619,0,0,5
9,9,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,False,316.799988,99.812386,400.949951,113.142387,0,1,0
10,10,False,10,TimesNewRomanPS-BoldItal,False,False,False,False,True,False,True,316.799988,123.752388,397.660065,137.142395,0,1,1
11,11,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,False,316.799988,147.812393,578.366577,221.142395,0,1,2
12,12,False,10,TimesNewRomanPS-BoldItal,False,False,False,False,True,False,True,316.799988,231.752380,375.980042,245.142395,0,1,3


In [10]:
# FOR DEBUGGING: Concatenated property matrix containing only mainbody and reference blocks from ALL pages before loop stops
allpages_mainbody_wizref_properties_df.head(3)

,index_in_page,is_firstline_indented,dominant_font_size,dominant_font_style,is_header,is_footer,is_footnote,is_reference,is_mainbody,is_sec_title,...,upperleft_x_coordinate,upperleft_y_coordinate,lowerright_x_coordinate,lowerright_y_coordinate,page_in_body,is_right_column,index_in_column,contents,section_num,section_name
0,3,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,...,36.0,100.163582,286.790039,149.303589,0,0,0,"Somalia, which is an East African country loca...",0,NaN
1,4,True,10,TimesNewRomanPSMT,False,False,False,False,True,False,...,36.0,148.163574,293.740051,305.303619,0,0,1,Civil war continued in Somalia in 2007. Ethiop...,0,NaN
2,5,False,10,TimesNewRomanPS-BoldMT,False,False,False,False,True,True,...,36.0,315.973602,83.600006,329.303619,0,0,2,Production\n,1,Production\n


#### Sec. 3: Retain blocks affiliated to *Government Policies and Programs* section from the PDF, and join after prettifying plain text extracted

In [8]:
## Extract 'Government Policies and Programs' Section from the PDF file, and deal with paragraphs span across columns
# Only retain blocks belonging to 'Government Policies and Programs' Section
govpolicy_blocks_df = allpages_mainbody_wizref_properties_df[allpages_mainbody_wizref_properties_df['section_name'].str.contains("Government Policies")]
# Initialize
govpolicy = str()
is_firstblock = True

for _, block in govpolicy_blocks_df.query("not is_sec_title").iterrows(): # Remove title before construct an iterator
    current_nolinebreak = block['contents'].replace('\n', '') # Remove all line breaks "\n" in the curent block
    if is_firstblock: # Create variable 'govpolicy' for saving items
        govpolicy = current_nolinebreak
        is_firstblock = False
    else:
        if not block['is_firstline_indented'] and not block['is_sec_subtitle']:
            govpolicy = govpolicy + ' ' + current_nolinebreak
        else:
            govpolicy = govpolicy + '\n' + current_nolinebreak

In [9]:
# FOR DEBUGGING: Check the prettified joined plain text
print(govpolicy)

Not necessary to run the following block in regular debugging:

In [158]:
## Save the prettified plain text to TXT file
txt_dir = "../../data/policy_section_txt/test/"
txt_name = pdf_names[i].split('.')[0] + "_policy.txt"
txt_path = os.path.join(txt_dir, txt_name)
with open(txt_path, 'w', encoding='utf-8') as file:
    file.write(govpolicy)